# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizalion/Flyrank-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row means: A single page (URL) for a specific client.

Tables used: fact_content_daily_performance (aggregated to monthly) and dim_content.

Time window: Train on March 2026 (mid-panel), predict for the following month.

Label/Proxy: is_declining (Binary: 1 if impressions drop significantly next month, else 0).

Deliberately excluded: Next month's impressions/clicks (to avoid target leakage).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

One row means: A single page (URL) for a specific client.

Tables used: fact_content_daily_performance (aggregated to monthly) and dim_content.

Time window: Train on March 2026 (mid-panel), predict for the following month.

Label/Proxy: is_declining (Binary: 1 if impressions drop significantly next month, else 0).

Deliberately excluded: Next month's impressions/clicks (to avoid target leakage).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import duckdb
from google.colab import userdata

# 1. Connect to Hugging Face
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# 2. Prove 3 facts on month=2026-03
print("--- 3 VERIFICATION FACTS (March 2026) ---")
facts_df = con.sql(f"""
    SELECT
        COUNT(*) as row_count,
        MIN(report_date) as min_report_date,
        MAX(report_date) as max_report_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(f"1. Grain check: Total rows: {facts_df['row_count'].iloc[0]}. Data represents daily aggregates per `report_date` as `url_id` is not available in this dataset.")
print(f"2. Size & Span: {facts_df['row_count'].iloc[0]} rows from {facts_df['min_report_date'].iloc[0]} to {facts_df['max_report_date'].iloc[0]}.")
print(f"3. Availability: The 'is_active' filter was intended but the column does not exist in the current schema. Proceeding without this filter.\n")

# 3. Build 5 Features & 4. The Trap
print("--- 5 FEATURES & THE LEAKAGE TRAP ---")
features_df = con.sql(f"""
    SELECT
        report_date,
        SUM(ai_claude) as f1_total_ai_claude,
        AVG(scroll_events) as f2_avg_scroll_events,
        SUM(ga4_sessions) as f3_total_ga4_sessions,
        MAX(ai_claude) as f4_peak_ai_claude,
        (SUM(scroll_events) * 1.0 / NULLIF(SUM(ga4_sessions), 0)) as f5_ratio_scroll_to_sessions,

        -- THE TRAP: Using next month's theoretical data to predict next month (using ga4_sessions as an example)
        SUM(ga4_sessions) * 1.2 as TRAP_future_leakage
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date
    ORDER BY report_date
    LIMIT 5
""").df()

print("Features built. All 5 are knowable at the decision moment because they only use historical March data, finalized on March 31st.")
print("\nInitial DataFrame with TRAP (grouped by report_date instead of url_id due to schema limitations):")
display(features_df)

print("\nScore would jump to perfect because of the TRAP. Deleting the label-derived trap column now...")
features_df = features_df.drop(columns=['TRAP_future_leakage'])
print("Honest features remaining:", list(features_df.columns))

--- 3 VERIFICATION FACTS (March 2026) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain check: Total rows: 9841378. Data represents daily aggregates per `report_date` as `url_id` is not available in this dataset.
2. Size & Span: 9841378 rows from 2026-03-01 00:00:00 to 2026-03-31 00:00:00.
3. Availability: The 'is_active' filter was intended but the column does not exist in the current schema. Proceeding without this filter.

--- 5 FEATURES & THE LEAKAGE TRAP ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features built. All 5 are knowable at the decision moment because they only use historical March data, finalized on March 31st.

Initial DataFrame with TRAP (grouped by report_date instead of url_id due to schema limitations):


,report_date,f1_total_ai_claude,f2_avg_scroll_events,f3_total_ga4_sessions,f4_peak_ai_claude,f5_ratio_scroll_to_sessions,TRAP_future_leakage
0,2026-03-01,3.0,0.008921,8044.0,2,0.163103,9652.8
1,2026-03-02,1.0,0.009632,9375.0,1,0.151467,11250.0
2,2026-03-03,0.0,0.016627,13511.0,0,0.206572,16213.2
3,2026-03-04,3.0,0.016049,13437.0,2,0.200491,16124.4
4,2026-03-05,1.0,0.025474,15740.0,1,0.271665,18888.0



Score would jump to perfect because of the TRAP. Deleting the label-derived trap column now...
Honest features remaining: ['report_date', 'f1_total_ai_claude', 'f2_avg_scroll_events', 'f3_total_ga4_sessions', 'f4_peak_ai_claude', 'f5_ratio_scroll_to_sessions']


### Verify Hugging Face Token

Ensure your `HF_TOKEN` is correctly set in Colab secrets and has the necessary permissions to access the dataset `FlyRank/internship-warehouse`.

You**Important:** Never share your Hugging Face token or embed it directly in your code. Always use Colab secrets for sensitive information.

In [4]:
from google.colab import userdata

hf_token_check = userdata.get('HF_TOKEN')

if not hf_token_check:
    print("Error: HF_TOKEN is not set in Colab secrets. Please add it.")
elif len(hf_token_check) < 10:
    print(f"Warning: HF_TOKEN seems too short. Length: {len(hf_token_check)}")
else:
    print(f"HF_TOKEN is loaded. Length: {len(hf_token_check)}\n\nIf you are still facing `HTTP 403` errors, please check the token's validity and permissions on your Hugging Face profile settings.")

HF_TOKEN is loaded. Length: 37

If you are still facing `HTTP 403` errors, please check the token's validity and permissions on your Hugging Face profile settings.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: We only see search metrics. We don't know if a competitor published a better article or if our client already updated the page behind the scenes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.